# Ingestion coverage — code-review-graph on `voice-assessment-agent`

Voice agent (Python only). Fills the team's `Ingestion_Coverage.xlsx` for this project.
**Functions expected** come from Python's own parser, `ast` (`ref_python.py`).

Every run is a new iteration in `results/voice-assessment-agent/runs/iter-NNN_<date>_<time>/` and adds
lines to `results/voice-assessment-agent/runs.csv`; nothing from an earlier run is overwritten.

Run: `./run.sh voice-assessment-agent` (from `notebooks/`), or Run All here and Save As into the iteration folder.

In [6]:
APP = 'voice-assessment-agent'   # this notebook's project (folder under apps/)
RUN = ''        # iteration folder name; empty = next free iter-NNN (run.sh sets it)

In [7]:
import sys; sys.path.insert(0, '.')
import pandas as pd
import coverage_lab as C
import ref_python
REFERENCES = [ref_python]   # this project's languages that get a functions-expected count
pd.set_option('display.max_rows', 300); pd.set_option('display.max_colwidth', 120)
RUN = RUN or C.next_run_id(APP)
OUT = C.run_dir(APP, RUN)   # results/<APP>/runs/<RUN>/
stamp = C.run_stamp(APP)
{**stamp, 'run': RUN}

$ git rev-parse --short HEAD 
$ /Users/thiru/.local/bin/code-review-graph --version 


{'app': 'voice-assessment-agent',
 'app_commit': '4e24898',
 'tool': 'code-review-graph 2.3.8',
 'run_at': '2026-11-17T20:38:45',
 'run': 'iter-003_2026-11-17_2038'}

## 1. Build the graph exactly as Synthetix does
`CRG_DATA_DIR=<dir> code-review-graph build --repo <root> -q`, after removing the repo from the tool's registry.

In [8]:
db = C.build(APP, OUT)
db

$ /Users/thiru/.local/bin/code-review-graph repos 
$ /Users/thiru/.local/bin/code-review-graph build --repo /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/apps/voice-assessment-agent -q (env {'CRG_DATA_DIR': '/Users/thiru/MySpace/Celsior/repo/code-graph-rnd/results/voice-assessment-agent/runs/iter-003_2026-11-17_2038/crg'})


PosixPath('/Users/thiru/MySpace/Celsior/repo/code-graph-rnd/results/voice-assessment-agent/runs/iter-003_2026-11-17_2038/crg/graph.db')

## 2. Sheet 1 — one row per code file
Read? / Why come from the tool's own filter and a re-parse with the same grammar.

In [9]:
rows, funcs, left_out = C.coverage(APP, db, REFERENCES)
sheet1 = pd.DataFrame([{k: v for k, v in r.__dict__.items() if k != 'expected'} for r in rows])
sheet1.read.value_counts()

$ git ls-files 


read
Fully read    74
Not read       3
Name: count, dtype: int64

### Files left out of sheet 1 (not code)
No language in the tool and not a programming-language extension: docs, licences, images, Dockerfiles, config.

In [10]:
print(len(left_out), 'files left out')
pd.Series([p.rsplit('.', 1)[-1] if '.' in p.rsplit('/', 1)[-1] else '(no extension)' for p in left_out]).value_counts()

41 files left out


md                27
(no extension)     4
json               2
txt                2
dockerignore       1
example            1
gitignore          1
agent              1
toml               1
html               1
Name: count, dtype: int64

In [11]:
sheet1[sheet1.read != 'Fully read'][['file', 'language', 'lines', 'read', 'why', 'lines_read', 'notes']]

,file,language,lines,read,why,lines_read,notes
0,.github/workflows/ci.yml,yaml,79.0,Not read,Language dropped,0,"passes the filter, parser returns nothing (parser.py:2674-2697)"
46,docker-compose.prod.yml,yaml,86.0,Not read,Language dropped,0,"passes the filter, parser returns nothing (parser.py:2674-2697)"
47,docker-compose.yml,yaml,89.0,Not read,Language dropped,0,"passes the filter, parser returns nothing (parser.py:2674-2697)"


## 3. Functions: expected (language's own parser) vs found (graph.db)

In [12]:
matched = C.match_functions(rows, funcs)
matched.groupby(['language', 'status']).size()

language  status                                         
python    found                                              477
          lost: same name stored once (graph.py:79, :243)     10
dtype: int64

In [13]:
matched[matched.status != 'found']

,language,file,function,line,status
260,python,tests/integration/test_assessment.py,_boom,182,"lost: same name stored once (graph.py:79, :243)"
344,python,tests/integration/test_telephony.py,_call,447,"lost: same name stored once (graph.py:79, :243)"
366,python,tests/unit/test_agent_tools_retry.py,mock_post,38,"lost: same name stored once (graph.py:79, :243)"
367,python,tests/unit/test_agent_tools_retry.py,mock_post,56,"lost: same name stored once (graph.py:79, :243)"
368,python,tests/unit/test_agent_tools_retry.py,mock_post,73,"lost: same name stored once (graph.py:79, :243)"
369,python,tests/unit/test_agent_tools_retry.py,mock_post,89,"lost: same name stored once (graph.py:79, :243)"
370,python,tests/unit/test_agent_tools_retry.py,mock_post,106,"lost: same name stored once (graph.py:79, :243)"
371,python,tests/unit/test_agent_tools_retry.py,mock_post,121,"lost: same name stored once (graph.py:79, :243)"
372,python,tests/unit/test_agent_tools_retry.py,mock_post,138,"lost: same name stored once (graph.py:79, :243)"
408,python,tests/unit/test_grading_hardening.py,create,15,"lost: same name stored once (graph.py:79, :243)"


### Files the reference parser could not fully check
Listed, never silently dropped (a syntax error, or code the reference cannot parse).

In [14]:
C.could_not_check(rows)

,language,file,reason


### Unnamed functions (callbacks, lambdas) — not counted as expected
Shown per file with the lines they cover. Their code is still walked by the tool; only a separate row is missing.

In [15]:
cb = C.callbacks_table(rows)
print(cb.callbacks.sum(), 'unnamed functions,', cb.lines.sum(), 'lines')
cb.sort_values('callbacks', ascending=False)

51 unnamed functions, 78 lines


,language,file,callbacks,lines
7,python,tests/unit/test_grading_hardening.py,22,22
5,python,tests/integration/test_grading_worker.py,8,22
6,python,tests/integration/test_telephony.py,7,7
0,python,app/agent/pipeline.py,3,9
3,python,tests/integration/test_assessment.py,3,10
4,python,tests/integration/test_evaluation.py,2,2
9,python,tests/unit/test_input_validation.py,2,2
1,python,app/rag/retriever.py,1,1
2,python,app/services/candidate_resolver.py,1,1
8,python,tests/unit/test_health_endpoints.py,1,1


## 4. Sheet 2 — per language

In [16]:
sheet2 = C.parsers_table(APP, rows, matched, RUN, REFERENCES)
sheet2

,Language,Package,Repo,Files tried,Files parsed,Functions expected,Functions found,Evidence,Notes
0,bash,code-review-graph 2.3.8 (grammar: bash),voice-assessment-agent,2,2,NaN,NaN,results/voice-assessment-agent/runs/iter-003_2026-11-17_2038/coverage.ipynb,no reference parser for this language yet
1,python,code-review-graph 2.3.8 (grammar: python),voice-assessment-agent,72,72,487.0,477.0,results/voice-assessment-agent/runs/iter-003_2026-11-17_2038/coverage.ipynb,expected from Python ast (stdlib); lost to same-name overwrite: 10; not found: 0; unnamed functions not counted: 51 ...
2,yaml,code-review-graph 2.3.8 (grammar: yaml),voice-assessment-agent,3,0,NaN,NaN,results/voice-assessment-agent/runs/iter-003_2026-11-17_2038/coverage.ipynb,no reference parser for this language yet


## 5. Sheet 3 — summary (same formulas as the workbook)

In [17]:
s = sheet1
summary = {'Files': len(s), 'Fully read': (s.read == 'Fully read').sum(), 'Partly read': (s.read == 'Partly read').sum(),
  'Not read': (s.read == 'Not read').sum(), 'Lines': int(s.lines.fillna(0).sum()), 'Lines read': int(s.lines_read.fillna(0).sum()),
  'Not shown to user': ((s.read != 'Fully read') & (s.shown == 'No')).sum()}
summary['% lines read'] = round(100 * summary['Lines read'] / summary['Lines'], 1)
summary

{'Files': 77,
 'Fully read': np.int64(74),
 'Partly read': np.int64(0),
 'Not read': np.int64(3),
 'Lines': 10274,
 'Lines read': 10020,
 'Not shown to user': np.int64(3),
 '% lines read': 97.5}

In [18]:
s[s.read == 'Not read'].why.value_counts()

why
Language dropped    3
Name: count, dtype: int64

## 6. Write the workbook

In [19]:
C.write_workbook(APP, rows, sheet2, OUT)

/Users/thiru/MySpace/Celsior/repo/code-graph-rnd/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


PosixPath('/Users/thiru/MySpace/Celsior/repo/code-graph-rnd/results/voice-assessment-agent/runs/iter-003_2026-11-17_2038/Ingestion_Coverage_voice-assessment-agent.xlsx')

## 7. Add this run to the repo's run log
`results/<repo>/runs.csv`, append-only: one line per run (per language with a reference parser).

In [20]:
log = C.log_run(APP, RUN, stamp, summary, sheet2)
pd.read_csv(log)

,run,run_at,app_commit,tool,files,fully_read,partly_read,not_read,lines,lines_read,pct_lines_read,language,functions_expected,functions_found,notes
0,iter-001_2026-11-17_1647,2026-11-17T16:47:44,inf,code-review-graph 2.3.8,77,74,0,3,10274,10020,97.5,python,487.0,477.0,"run by Claude before the iteration layout; executed notebook not kept, see runs/iter-001_2026-11-17_1647/NOTE.md"
1,iter-002_2026-11-17_1858,2026-11-17T18:58:40,inf,code-review-graph 2.3.8,77,74,0,3,10274,10020,97.5,python,487.0,477.0,NaN
2,iter-003_2026-11-17_2038,2026-11-17T20:38:45,inf,code-review-graph 2.3.8,77,74,0,3,10274,10020,97.5,bash,NaN,NaN,no reference parser for this language yet
3,iter-003_2026-11-17_2038,2026-11-17T20:38:45,inf,code-review-graph 2.3.8,77,74,0,3,10274,10020,97.5,python,487.0,477.0,NaN
4,iter-003_2026-11-17_2038,2026-11-17T20:38:45,inf,code-review-graph 2.3.8,77,74,0,3,10274,10020,97.5,yaml,NaN,NaN,no reference parser for this language yet
